# Notebook 06 — v2 LOB Classifier

**Goal:** Design v2 classifier with payer_name signals, dental/vision exclusion, and rate-value anchor flags. Target: drop `unknown` to <10% per hospital.

**Phase 1 (this session):** Bottom-up — enumerate distinct payer_name values in v1's `unknown` bucket across all 5 hospitals.

**Phase 2 (next session):** Draft GOVERNMENT_PAYER_PATTERNS and SPECIALTY_PAYER_PATTERNS from phase 1 output. Implement asymmetric override logic and rate_anchor_match flag.

**Phase 3:** Apply v2, verify unknown <10%, re-run notebook 05 charts.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import duckdb

from src.queries import classify_plan, add_lob, query_procedure_rates_agg, LOB_PATTERNS

print("Imports OK")
print(f"LOB_PATTERNS has {len(LOB_PATTERNS)} rules")

Imports OK
LOB_PATTERNS has 9 rules


In [3]:
RAW = Path('../data/raw')

hospitals = {
    'baylor':                RAW / 'baylor_university_medical_center-69947_parsed.duckdb',
    'methodist':             RAW / 'methodist_dallas_medical_center-6000b_parsed.duckdb',
    'parkland':              RAW / 'parkland_health-6e88d_parsed.duckdb',
    'texas_health_plano':    RAW / 'texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb',
    'medical_city_alliance': RAW / 'medical_city_alliance_hospital-77912_parsed.duckdb',
}

for name, path in hospitals.items():
    print(f"{name:25s} {'OK' if path.exists() else 'MISSING'}  {path.name}")

cons = {name: duckdb.connect(str(path), read_only=True) for name, path in hospitals.items()}

print(f"\nOpened {len(cons)} connections.")

baylor                    OK  baylor_university_medical_center-69947_parsed.duckdb
methodist                 OK  methodist_dallas_medical_center-6000b_parsed.duckdb
parkland                  OK  parkland_health-6e88d_parsed.duckdb
texas_health_plano        OK  texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb
medical_city_alliance     OK  medical_city_alliance_hospital-77912_parsed.duckdb

Opened 5 connections.


In [4]:
dfs = {}
for name, con in cons.items():
    df = query_procedure_rates_agg(con, '73721')
    df = add_lob(df)
    dfs[name] = df
    
    n_total = len(df)
    n_unknown = (df['lob'] == 'unknown').sum()
    pct_unknown = 100 * n_unknown / n_total if n_total else 0
    print(f"  {name:25s} {n_total:4d} combos   {n_unknown:4d} unknown ({pct_unknown:5.1f}%)")

  baylor                      45 combos     19 unknown ( 42.2%)
  methodist                   82 combos     46 unknown ( 56.1%)
  parkland                    65 combos     16 unknown ( 24.6%)
  texas_health_plano          40 combos     12 unknown ( 30.0%)
  medical_city_alliance       46 combos     34 unknown ( 73.9%)


In [5]:
print("Distinct payer_names in v1 'unknown' bucket, per hospital:\n")

for name, df in dfs.items():
    unknown_df = df[df['lob'] == 'unknown']
    payer_counts = (
        unknown_df
        .groupby('payer_name', dropna=False)
        .size()
        .sort_values(ascending=False)
    )
    
    print(f"=== {name} ({len(unknown_df)} unknown combos, {payer_counts.shape[0]} distinct payers) ===")
    for payer, count in payer_counts.items():
        payer_display = payer if pd.notna(payer) else '<NULL>'
        print(f"  {count:3d}  {payer_display}")
    print()

Distinct payer_names in v1 'unknown' bucket, per hospital:

=== baylor (19 unknown combos, 11 distinct payers) ===
    5  Baylor Scott & White Health Plan
    3  Blue Cross Blue Shield
    2  HealthSmart
    2  United Healthcare
    1  CORVEL
    1  PHCS
    1  Cigna
    1  Prime Health Services
    1  QuickTrip
    1  Texas Workforce Commission
    1  TriWest

=== methodist (46 unknown combos, 37 distinct payers) ===
    5  BCBS [3001]
    3  AETNA MANAGED CARE [2068]
    2  AETNA COMMERCIAL [2042]
    2  AETNA [3000]
    2  WEB TPA [3049]
    1  AETNA TRANSPLANT NETWORK [3025]
    1  AETNA MEDICARE MANAGED CARE [7000]
    1  90 DEGREE BENEFIT [3057]
    1  ALTERNATIVE SERVICE CONCEPTS [8005]
    1  AMBETTER SUPERIOR [3051]
    1  BRIGHT HEALTHCARE [2222]
    1  CIGNA MEDICARE [7011]
    1  ALLIED NATIONAL [2031]
    1  COVID19 HRSA UNINSURED TESTING AND TREATMENT FUND [6005]
    1  DART MEMBER CARE [3039]
    1  DEPARTMENT OF LABOR [8004]
    1  ENTRUST/90 DEGREE [2058]
    1  EVRY H

In [6]:


output_path = Path('../outputs/unknown_payers_v1.txt')
output_path.parent.mkdir(exist_ok=True)

lines = []
lines.append("Distinct payer_names in v1 'unknown' bucket, per hospital")
lines.append("=" * 70)
lines.append("")

for name, df in dfs.items():
    unknown_df = df[df['lob'] == 'unknown']
    payer_counts = (
        unknown_df
        .groupby('payer_name', dropna=False)
        .size()
        .sort_values(ascending=False)
    )
    
    lines.append(f"=== {name} ({len(unknown_df)} unknown combos, {payer_counts.shape[0]} distinct payers) ===")
    for payer, count in payer_counts.items():
        payer_display = payer if pd.notna(payer) else '<NULL>'
        lines.append(f"  {count:3d}  {payer_display}")
    lines.append("")

output_path.write_text("\n".join(lines))
print(f"Wrote {len(lines)} lines to {output_path}")
print(f"\nFirst 30 lines preview:\n")
print("\n".join(lines[:30]))

Wrote 93 lines to ..\outputs\unknown_payers_v1.txt

First 30 lines preview:

Distinct payer_names in v1 'unknown' bucket, per hospital

=== baylor (19 unknown combos, 11 distinct payers) ===
    5  Baylor Scott & White Health Plan
    3  Blue Cross Blue Shield
    2  HealthSmart
    2  United Healthcare
    1  CORVEL
    1  PHCS
    1  Cigna
    1  Prime Health Services
    1  QuickTrip
    1  Texas Workforce Commission
    1  TriWest

=== methodist (46 unknown combos, 37 distinct payers) ===
    5  BCBS [3001]
    3  AETNA MANAGED CARE [2068]
    2  AETNA COMMERCIAL [2042]
    2  AETNA [3000]
    2  WEB TPA [3049]
    1  AETNA TRANSPLANT NETWORK [3025]
    1  AETNA MEDICARE MANAGED CARE [7000]
    1  90 DEGREE BENEFIT [3057]
    1  ALTERNATIVE SERVICE CONCEPTS [8005]
    1  AMBETTER SUPERIOR [3051]
    1  BRIGHT HEALTHCARE [2222]
    1  CIGNA MEDICARE [7011]
    1  ALLIED NATIONAL [2031]


In [7]:


methodist = dfs['methodist']
mask = (methodist['lob'] == 'unknown') & methodist['payer_name'].str.contains('MEDICARE', case=False, na=False)
suspects = methodist[mask]

print(f"Methodist unknown rows with 'MEDICARE' in payer_name: {len(suspects)}")
print()

for idx, row in suspects.iterrows():
    print(f"--- row index {idx} ---")
    for col in suspects.columns:
        val = row[col]
        val_display = repr(val) if isinstance(val, str) else val
        print(f"  {col:30s} {val_display}")
    print()

Methodist unknown rows with 'MEDICARE' in payer_name: 5

--- row index 29 ---
  payer_name                     'MULTIPLAN MEDICARE MANAGED CARE [7022]'
  plan_name                      'MHS HB MULTIPLAN ADVANTAGE MDMC'
  payer_group                    'Medicare'
  payer_type                     'Medicare'
  methodology_normalized         'fee schedule'
  billing_class_normalized       None
  n_charge_ids                   1
  dollar_rate                    232.47
  dollar_min                     232.47
  dollar_max                     232.47
  pct_rate                       nan
  gross_min                      4878.0
  gross_max                      4878.0
  descriptions                   'HC MRI Extremity Lwr Joint Woc'
  lob                            'unknown'
  lob_rule                       'no_match'

--- row index 35 ---
  payer_name                     'AETNA MEDICARE MANAGED CARE [7000]'
  plan_name                      'MHS HB AETNA GOLDEN MDMC'
  payer_group                 

In [8]:


for name, df in dfs.items():
    print(f"=== {name} (n={len(df)}) ===")
    
    for col in ['payer_group', 'payer_type']:
        n_null = df[col].isna().sum()
        n_populated = len(df) - n_null
        pct_populated = 100 * n_populated / len(df) if len(df) else 0
        
        print(f"  {col}: {n_populated}/{len(df)} populated ({pct_populated:.0f}%), "
              f"{df[col].nunique(dropna=True)} distinct values")
        
      
        cross = (
            df.groupby([col, 'lob'], dropna=False)
              .size()
              .unstack(fill_value=0)
        )
       
        cross['_total'] = cross.sum(axis=1)
        cross = cross.sort_values('_total', ascending=False).drop(columns='_total')
        
        print(cross.to_string())
        print()
    print()

=== baylor (n=45) ===
  payer_group: 45/45 populated (100%), 7 distinct values
lob               aca_exchange  commercial  medicaid_chip  medicaid_other  medicare_advantage  medicare_traditional  unknown
payer_group                                                                                                                 
Other                        1           6              1               1                   3                     2       13
BCBS                         0           2              0               0                   1                     0        3
UnitedHealthcare             0           1              0               0                   1                     0        2
Cigna                        0           1              0               0                   1                     0        1
Aetna                        0           1              0               0                   1                     0        0
Humana                       0           1    

In [9]:


output_path = Path('../outputs/payer_columns_diagnostic.txt')
output_path.parent.mkdir(exist_ok=True)

lines = []
lines.append("payer_group and payer_type vs v1 lob classification, per hospital")
lines.append("=" * 70)
lines.append("")

for name, df in dfs.items():
    lines.append(f"=== {name} (n={len(df)}) ===")
    
    for col in ['payer_group', 'payer_type']:
        n_null = df[col].isna().sum()
        n_populated = len(df) - n_null
        pct_populated = 100 * n_populated / len(df) if len(df) else 0
        
        lines.append(f"  {col}: {n_populated}/{len(df)} populated ({pct_populated:.0f}%), "
                     f"{df[col].nunique(dropna=True)} distinct values")
        
        cross = (
            df.groupby([col, 'lob'], dropna=False)
              .size()
              .unstack(fill_value=0)
        )
        cross['_total'] = cross.sum(axis=1)
        cross = cross.sort_values('_total', ascending=False).drop(columns='_total')
        
        lines.append(cross.to_string())
        lines.append("")
    lines.append("")

output_path.write_text("\n".join(lines))
print(f"Wrote {len(lines)} lines to {output_path}")

Wrote 43 lines to ..\outputs\payer_columns_diagnostic.txt


In [10]:
methodist_df = dfs['methodist']  

medicare_unknowns = methodist_df[
    (methodist_df['payer_group'] == 'Medicare') &
    (methodist_df['lob'] == 'unknown')
]

molina_ma = methodist_df[
    (methodist_df['payer_group'] == 'Molina') &
    (methodist_df['lob'] == 'medicare_advantage')
]

cols = ['payer_name', 'plan_name', 'payer_type', 'payer_group',
        'dollar_rate', 'lob', 'lob_rule']

print("=== Methodist payer_group='Medicare' unknowns ===")
print(medicare_unknowns[cols].to_string())
print(f"\n=== Methodist payer_group='Molina' medicare_advantage ===")
print(molina_ma[cols].to_string())

=== Methodist payer_group='Medicare' unknowns ===
                                      payer_name                        plan_name payer_type payer_group  dollar_rate      lob  lob_rule
29        MULTIPLAN MEDICARE MANAGED CARE [7022]  MHS HB MULTIPLAN ADVANTAGE MDMC   Medicare    Medicare       232.47  unknown  no_match
53  MUTUAL OF OMAHA MEDICARE MANAGED CARE [7019]      MHS HB MUTUAL OF OMAHA MDMC   Medicare    Medicare       232.47  unknown  no_match

=== Methodist payer_group='Molina' medicare_advantage ===
Empty DataFrame
Columns: [payer_name, plan_name, payer_type, payer_group, dollar_rate, lob, lob_rule]
Index: []


In [11]:
print("\n=== v1 LOB_PATTERNS (in order) ===\n")
from src.queries import LOB_PATTERNS
print(f"type: {type(LOB_PATTERNS).__name__}")
print(f"len:  {len(LOB_PATTERNS)}\n")
print(f"first element type: {type(LOB_PATTERNS[0]).__name__}")
print(f"first element:      {LOB_PATTERNS[0]}\n")
for i, item in enumerate(LOB_PATTERNS):
    print(f"{i:2d}  {item}")


=== v1 LOB_PATTERNS (in order) ===

type: list
len:  9

first element type: tuple
first element:      ('\\bCHIP\\b', 'medicaid_chip', 'plan_contains_chip')

 0  ('\\bCHIP\\b', 'medicaid_chip', 'plan_contains_chip')
 1  ('STAR\\s*PLUS', 'medicaid_star_plus', 'plan_contains_star_plus')
 2  ('\\bSTAR\\b', 'medicaid_star', 'plan_contains_star')
 3  ('Medicaid', 'medicaid_other', 'plan_contains_medicaid')
 4  ('Medicare\\s*Advantage', 'medicare_advantage', 'plan_contains_medicare_advantage')
 5  ('Exchange|Marketplace|ACA', 'aca_exchange', 'plan_contains_aca_exchange')
 6  ('Medicare', 'medicare_traditional', 'plan_contains_medicare')
 7  ('Commercial', 'commercial', 'plan_contains_commercial')
 8  ('\\bPPO\\b|\\bHMO\\b|\\bEPO\\b', 'commercial', 'plan_contains_network_type')


In [12]:
print("=== THP payer_group='Molina' rows (full detail) ===\n")
thp_molina = dfs['texas_health_plano'][
    dfs['texas_health_plano']['payer_group'] == 'Molina'
]
cols = ['payer_name', 'plan_name', 'payer_type', 'dollar_rate', 'lob', 'lob_rule']
print(thp_molina[cols].to_string())

print("\n=== Baylor payer_group='Molina' row ===\n")
baylor_molina = dfs['baylor'][dfs['baylor']['payer_group'] == 'Molina']
print(baylor_molina[cols].to_string())

=== THP payer_group='Molina' rows (full detail) ===

   payer_name     plan_name  payer_type  dollar_rate                   lob                lob_rule
6      Molina  Medicare MMP  Commercial       235.35  medicare_traditional  plan_contains_medicare
26     Molina  Medicare HMO  Commercial       235.35  medicare_traditional  plan_contains_medicare

=== Baylor payer_group='Molina' row ===

   payer_name           plan_name  payer_type  dollar_rate                 lob                          lob_rule
14     Molina  Medicare Advantage  Commercial        235.3  medicare_advantage  plan_contains_medicare_advantage


In [13]:
def classify_plan_v2(row: pd.Series) -> tuple[str, str]:
    """Returns (lob, lob_rule). Reads row['plan_name'], row['payer_name'], row['payer_group']."""

In [14]:
from src.queries import classify_plan_v2

test_cases = [

    ("Methodist #17 (Multiplan Medicare MC) [known FN until 2.2]",
     'Medicare', 'MHS HB MULTIPLAN ADVANTAGE MDMC',
     'medicare_advantage', 'plan_medicare_qualified'),

    ("Methodist #44 (Mutual of Omaha MC) [known FN until 2.2]",
     'Medicare', 'MHS HB MUTUAL OF OMAHA MDMC',
     'medicare_advantage', 'plan_medicare_qualified'),

    ("THP Molina Medicare HMO",
     'Molina', 'Medicare HMO',
     'medicare_advantage', 'plan_medicare_qualified'),

    ("THP Molina Medicare MMP",
     'Molina', 'Medicare MMP',
     'medicare_advantage', 'plan_medicare_qualified'),

    ("Baylor Molina Medicare Advantage",
     'Molina', 'Medicare Advantage',
     'medicare_advantage', 'plan_medicare_advantage'),
]

print(f"{'case':<58} {'got lob':<22} {'got rule':<28} {'ok'}")
print("-" * 120)
for desc, pg, pn, exp_lob, exp_rule in test_cases:
    row = pd.Series({'plan_name': pn, 'payer_group': pg})
    got_lob, got_rule = classify_plan_v2(row)
    if (got_lob, got_rule) == (exp_lob, exp_rule):
        ok = "PASS"
    else:
        ok = f"FAIL  expected ({exp_lob}, {exp_rule})"
    print(f"{desc:<58} {got_lob:<22} {got_rule:<28} {ok}")

case                                                       got lob                got rule                     ok
------------------------------------------------------------------------------------------------------------------------
Methodist #17 (Multiplan Medicare MC) [known FN until 2.2] medicare_traditional   payer_group_medicare         FAIL  expected (medicare_advantage, plan_medicare_qualified)
Methodist #44 (Mutual of Omaha MC) [known FN until 2.2]    medicare_traditional   payer_group_medicare         FAIL  expected (medicare_advantage, plan_medicare_qualified)
THP Molina Medicare HMO                                    medicare_advantage     plan_medicare_qualified      PASS
THP Molina Medicare MMP                                    medicare_advantage     plan_medicare_qualified      PASS
Baylor Molina Medicare Advantage                           medicare_advantage     plan_medicare_advantage      PASS


In [15]:


from src.queries import add_lob_v2


dfs_v2 = {name: add_lob_v2(df) for name, df in dfs.items()}


print(f"{'hospital':<22} {'rows':>6} {'v1 unk':>7} {'v1 %':>6} {'v2 unk':>7} {'v2 %':>6} {'delta':>7}")
print("-" * 70)
total_rows = total_v1 = total_v2 = 0
for name in dfs:
    n = len(dfs[name])
    v1_unk = (dfs[name]['lob'] == 'unknown').sum()
    v2_unk = (dfs_v2[name]['lob'] == 'unknown').sum()
    v1_pct = 100 * v1_unk / n
    v2_pct = 100 * v2_unk / n
    delta  = v2_unk - v1_unk
    flag   = "" if v2_pct < 10 else "  ← above 10% target"
    print(f"{name:<22} {n:>6} {v1_unk:>7} {v1_pct:>5.1f}% {v2_unk:>7} {v2_pct:>5.1f}% {delta:>+7}{flag}")
    total_rows += n
    total_v1   += v1_unk
    total_v2   += v2_unk

print("-" * 70)
print(f"{'TOTAL':<22} {total_rows:>6} {total_v1:>7} {100*total_v1/total_rows:>5.1f}% {total_v2:>7} {100*total_v2/total_rows:>5.1f}% {total_v2 - total_v1:>+7}")

print("\n=== v2 lob_rule distribution (all hospitals combined) ===\n")
all_v2 = pd.concat([df.assign(hospital=name) for name, df in dfs_v2.items()], ignore_index=True)
rule_counts = all_v2.groupby('lob_rule').size().sort_values(ascending=False)
for rule, count in rule_counts.items():
    pct = 100 * count / len(all_v2)
    print(f"  {count:>4}  {pct:>5.1f}%  {rule}")

hospital                 rows  v1 unk   v1 %  v2 unk   v2 %   delta
----------------------------------------------------------------------
baylor                     45      19  42.2%      13  28.9%      -6  ← above 10% target
methodist                  82      46  56.1%      27  32.9%     -19  ← above 10% target
parkland                   65      16  24.6%       6   9.2%     -10
texas_health_plano         40      12  30.0%       1   2.5%     -11
medical_city_alliance      46      34  73.9%      12  26.1%     -22  ← above 10% target
----------------------------------------------------------------------
TOTAL                     278     127  45.7%      59  21.2%     -68

=== v2 lob_rule distribution (all hospitals combined) ===

    59   21.2%  no_match
    32   11.5%  plan_medicare
    23    8.3%  payer_group_commercial_aetna
    23    8.3%  payer_group_commercial_bcbs
    23    8.3%  plan_chip
    19    6.8%  plan_network_type
    15    5.4%  plan_medicare_advantage
    15    5.4%  pl

In [16]:


print("=== v2 unknowns by hospital x payer_group ===\n")
for name, df in dfs_v2.items():
    unk = df[df['lob'] == 'unknown']
    if len(unk) == 0:
        print(f"{name}: 0 unknowns")
        continue
    print(f"--- {name} ({len(unk)} unknowns) ---")
    pg_counts = unk.groupby('payer_group', dropna=False).size().sort_values(ascending=False)
    for pg, count in pg_counts.items():
        pg_display = pg if pd.notna(pg) else '<NULL>'
        print(f"  {count:3d}  {pg_display}")
    print()


print("\n=== v2 unknowns: distinct payer_names per hospital ===\n")
lines = []
for name, df in dfs_v2.items():
    unk = df[df['lob'] == 'unknown']
    if len(unk) == 0:
        continue
    lines.append(f"=== {name} ({len(unk)} unknowns) ===")

    grouped = unk.groupby(['payer_group', 'payer_name'], dropna=False).size().sort_values(ascending=False)
    for (pg, pn), count in grouped.items():
        pg_d = pg if pd.notna(pg) else '<NULL>'
        pn_d = pn if pd.notna(pn) else '<NULL>'
        lines.append(f"  {count:3d}  [{pg_d:>10s}]  {pn_d}")
    lines.append("")


from pathlib import Path
out_path = Path("../outputs/unknown_payers_v2.txt")
out_path.write_text("\n".join(lines))
print(f"Wrote {len(lines)} lines to {out_path}")
print(f"(also showing first 40 lines below)\n")
for line in lines[:40]:
    print(line)

=== v2 unknowns by hospital x payer_group ===

--- baylor (13 unknowns) ---
   13  Other

--- methodist (27 unknowns) ---
   27  Other

--- parkland (6 unknowns) ---
    6  Other

--- texas_health_plano (1 unknowns) ---
    1  Other

--- medical_city_alliance (12 unknowns) ---
   10  Other
    2  Molina


=== v2 unknowns: distinct payer_names per hospital ===

Wrote 58 lines to ..\outputs\unknown_payers_v2.txt
(also showing first 40 lines below)

=== baylor (13 unknowns) ===
    5  [     Other]  Baylor Scott & White Health Plan
    2  [     Other]  HealthSmart
    1  [     Other]  CORVEL
    1  [     Other]  PHCS
    1  [     Other]  Prime Health Services
    1  [     Other]  QuickTrip
    1  [     Other]  Texas Workforce Commission
    1  [     Other]  TriWest

=== methodist (27 unknowns) ===
    2  [     Other]  WEB TPA [3049]
    1  [     Other]  90 DEGREE BENEFIT [3057]
    1  [     Other]  ALTERNATIVE SERVICE CONCEPTS [8005]
    1  [     Other]  AMBETTER SUPERIOR [3051]
    1  [  

In [17]:
print("=== MCA payer_group='Molina' unknowns ===\n")
mca_molina_unk = dfs_v2['medical_city_alliance'][
    (dfs_v2['medical_city_alliance']['payer_group'] == 'Molina') &
    (dfs_v2['medical_city_alliance']['lob'] == 'unknown')
]
cols = ['payer_name', 'plan_name', 'payer_type', 'dollar_rate', 'lob', 'lob_rule']
print(mca_molina_unk[cols].to_string())

=== MCA payer_group='Molina' unknowns ===

   payer_name   plan_name  payer_type  dollar_rate      lob  lob_rule
25   "Molina"  "STARKIDS"  Commercial       231.81  unknown  no_match
32   "Molina"   "MCDSTAR"  Commercial       231.81  unknown  no_match


In [18]:

from pathlib import Path
print(Path("../outputs/unknown_payers_v2.txt").read_text())

=== baylor (13 unknowns) ===
    5  [     Other]  Baylor Scott & White Health Plan
    2  [     Other]  HealthSmart
    1  [     Other]  CORVEL
    1  [     Other]  PHCS
    1  [     Other]  Prime Health Services
    1  [     Other]  QuickTrip
    1  [     Other]  Texas Workforce Commission
    1  [     Other]  TriWest

=== methodist (27 unknowns) ===
    2  [     Other]  WEB TPA [3049]
    1  [     Other]  90 DEGREE BENEFIT [3057]
    1  [     Other]  ALTERNATIVE SERVICE CONCEPTS [8005]
    1  [     Other]  AMBETTER SUPERIOR [3051]
    1  [     Other]  BRIGHT HEALTHCARE [2222]
    1  [     Other]  COVID19 HRSA UNINSURED TESTING AND TREATMENT FUND [6005]
    1  [     Other]  DART MEMBER CARE [3039]
    1  [     Other]  DEPARTMENT OF LABOR [8004]
    1  [     Other]  ENTRUST/90 DEGREE [2058]
    1  [     Other]  ALLIED NATIONAL [2031]
    1  [     Other]  EVRY HEALTH [1013]
    1  [     Other]  FRIDAY HEALTH PLAN [2061]
    1  [     Other]  GLOBALHEALTH [1003]
    1  [     Other]  GENER

In [26]:
v3_test_cases = [
    # --- v2 cases (regression check — these should still pass under v3) ---
    ("v2 carry: THP Molina HMO",
     "Molina Healthcare", "Molina Medicare HMO", "Molina",
     "medicare_advantage", "plan_medicare_qualified"),
    ("v2 carry: THP Molina MMP",
     "Molina Healthcare", "Molina Medicare MMP", "Molina",
     "medicare_advantage", "plan_medicare_qualified"),
    ("v2 carry: Baylor Molina MA",
     "Molina Healthcare", "Molina Medicare Advantage", "Molina",
     "medicare_advantage", "plan_medicare_advantage"),

    # --- v3 NEW: Misclassified-government recovery ---
    ("v3 new: Methodist AMBETTER SUPERIOR → medicaid_chip (Decision 15)",
     "AMBETTER SUPERIOR [3051]", "MHS HB AMBETTER SUPERIOR MDMC", "Other",
     "medicaid_chip", "payer_name_ambetter_superior"),
    ("v3 new: Parkland CARE IMPROVEMENT PLUS → medicare_advantage",
     "CARE IMPROVEMENT PLUS [1104]", "Care Improvement Plus", "Other",
     "medicare_advantage", "payer_name_medicare_advantage_carrier"),
    ("v3 new: Parkland WELLCARE OF TEXAS → medicare_advantage",
     "WELLCARE OF TEXAS [1330]", "Wellcare of Texas", "Other",
     "medicare_advantage", "payer_name_medicare_advantage_carrier"),
    ("v3 new: MCA Amerigroup → medicaid_chip",
     "Amerigroup", "Amerigroup", "Other",
     "medicaid_chip", "payer_name_medicaid_carrier"),
    ("v3 new: Methodist WELLPOINT MARKETPLACE → aca_exchange (via plan_name MARKETPLACE keyword post-Decision-17)",
     "WELLPOINT MARKETPLACE EXCHANGE [3071]", "MHS HB WELLPOINT MARKETPLACE MDMC", "Other",
     "aca_exchange", "plan_aca"),
    ("v3 new: Methodist BRIGHT HEALTHCARE → aca_exchange (Decision 12, payer_name path)",
     "BRIGHT HEALTHCARE [2222]", "MHS HB BRIGHT HEALTHCARE MDMC", "Other",
     "aca_exchange", "payer_name_aca_exchange"),

    # --- v3 NEW: New LOB buckets (Decision 11) ---
    ("v3 new: Baylor CORVEL → workers_comp",
     "CORVEL", "CORVEL", "Other",
     "workers_comp", "payer_name_workers_comp"),
    ("v3 new: Methodist GENERIC WORKERS COMP TX → workers_comp",
     "GENERIC WORKERS COMP TX PAR [8002]", "MHS HB WORKERS COMP MDMC", "Other",
     "workers_comp", "payer_name_workers_comp"),
    ("v3 new: Parkland TRIWEST → federal_other",
     "TRIWEST HEALTHCARE ALLIANCE [1409]", "TriWest", "Other",
     "federal_other", "payer_name_federal_other"),
    ("v3 new: Baylor BSW Health Plan → employer_captive (Decision 13)",
     "Baylor Scott & White Health Plan", "BSW Health Plan", "Other",
     "employer_captive", "payer_name_employer_captive"),
    ("v3 new: MCA Parkland Community Health Plan → employer_captive",
     "Parkland Community Health Plan", "Parkland Community Health Plan", "Other",
     "employer_captive", "payer_name_employer_captive"),
    ("v3 new: Methodist WEB TPA → tpa_self_funded",
     "WEB TPA [3049]", "MHS HB WEB TPA MDMC", "Other",
     "tpa_self_funded", "payer_name_tpa"),
    ("v3 new: Baylor HealthSmart → tpa_self_funded",
     "HealthSmart", "HealthSmart", "Other",
     "tpa_self_funded", "payer_name_tpa"),

    # --- v3 NEW: Named-commercial-carrier recovery ---
    ("v3 new: MCA bare 'United' → commercial (anchored ^United$)",
     "United", "United", "Other",
     "commercial", "payer_name_commercial_carrier"),
    ("v3 new: THP ChoiceCare → commercial (Humana network brand)",
     "ChoiceCare", "ChoiceCare", "Other",
     "commercial", "payer_name_commercial_carrier"),
    ("v3 new: Parkland GREAT WEST → commercial (Cigna legacy)",
     "GREAT WEST HEALTHCARE [1037]", "Great West Healthcare", "Other",
     "commercial", "payer_name_commercial_carrier"),

    # --- v3 KNOWN FNs (still unresolved — Decision 16) ---
    ("v3 known FN (Decision 16): Methodist #17 MULTIPLAN ADVANTAGE — payer_group=Medicare short-circuits before payer_name pass",
     "MULTIPLAN MEDICARE MANAGED CARE [7022]", "MHS HB MULTIPLAN ADVANTAGE MDMC", "Medicare",
     "medicare_advantage", "payer_name_medicare_advantage_carrier"),
    ("v3 known FN (Decision 16): Methodist #44 MUTUAL OF OMAHA — payer_group=Medicare short-circuits",
     "MUTUAL OF OMAHA MEDICARE [7099]", "MHS HB MUTUAL OF OMAHA MDMC", "Medicare",
     "medicare_advantage", "payer_name_medicare_advantage_carrier"),
]

In [27]:

import pandas as pd
from src.queries import classify_plan_v3

print(f"=== v3 smoke test ({len(v3_test_cases)} cases) ===\n")
n_pass = n_fail = n_known_fn = 0

for desc, payer_name, plan_name, payer_group, expected_lob, expected_rule_substr in v3_test_cases:
    row = pd.Series({
        'payer_name': payer_name,
        'plan_name': plan_name,
        'payer_group': payer_group,
    })
    actual_lob, actual_rule = classify_plan_v3(row)

    lob_match = (actual_lob == expected_lob)
    rule_match = (expected_rule_substr in actual_rule)
    passed = lob_match and rule_match

    is_known_fn = "known FN" in desc
    if passed:
        status = "PASS"
        n_pass += 1
    elif is_known_fn:
        status = "FAIL (expected — known FN)"
        n_known_fn += 1
    else:
        status = "FAIL"
        n_fail += 1

    print(f"  [{status}]")
    print(f"    {desc}")
    print(f"    expected: ({expected_lob}, *{expected_rule_substr}*)")
    print(f"    actual:   ({actual_lob}, {actual_rule})")
    print()

print(f"Summary: {n_pass} PASS, {n_fail} FAIL (unexpected), {n_known_fn} FAIL (known FN)")

=== v3 smoke test (21 cases) ===

  [PASS]
    v2 carry: THP Molina HMO
    expected: (medicare_advantage, *plan_medicare_qualified*)
    actual:   (medicare_advantage, plan_medicare_qualified)

  [PASS]
    v2 carry: THP Molina MMP
    expected: (medicare_advantage, *plan_medicare_qualified*)
    actual:   (medicare_advantage, plan_medicare_qualified)

  [PASS]
    v2 carry: Baylor Molina MA
    expected: (medicare_advantage, *plan_medicare_advantage*)
    actual:   (medicare_advantage, plan_medicare_advantage)

  [PASS]
    v3 new: Methodist AMBETTER SUPERIOR → medicaid_chip (Decision 15)
    expected: (medicaid_chip, *payer_name_ambetter_superior*)
    actual:   (medicaid_chip, payer_name_ambetter_superior)

  [PASS]
    v3 new: Parkland CARE IMPROVEMENT PLUS → medicare_advantage
    expected: (medicare_advantage, *payer_name_medicare_advantage_carrier*)
    actual:   (medicare_advantage, payer_name_medicare_advantage_carrier)

  [PASS]
    v3 new: Parkland WELLCARE OF TEXAS → medic

In [21]:
import importlib
import src.queries
importlib.reload(src.queries)
from src.queries import classify_plan_v3
print('ok')

ok


In [22]:
# Day 9: Apply v3 to all_v2 (which already has v2 lob/lob_rule), compare, characterize residual.

import importlib, src.queries
importlib.reload(src.queries)
from src.queries import add_lob_v3

# all_v2 has v2's lob/lob_rule already populated. Apply v3 to a clean copy.
# We use the same source columns (payer_name, plan_name, payer_group) — v3 reads
# only those, so the existing lob/lob_rule will get overwritten by add_lob_v3.

all_v3 = add_lob_v3(all_v2.drop(columns=['lob', 'lob_rule']))

# Re-attach v2's classifications for side-by-side comparison
all_v3['lob_v2'] = all_v2['lob'].values
all_v3['lob_rule_v2'] = all_v2['lob_rule'].values
all_v3 = all_v3.rename(columns={'lob': 'lob_v3', 'lob_rule': 'lob_rule_v3'})

# --- 1. Per-hospital v2 vs v3 unknown comparison table ---

print("=== Per-hospital unknown counts: v2 vs v3 ===\n")
print(f"{'hospital':<22} {'total':>6} {'v2_unk':>8} {'v2_%':>7} {'v3_unk':>8} {'v3_%':>7} {'delta':>7} {'<10%?':>6}")
print("-" * 82)

for name, group in all_v3.groupby('hospital'):
    total = len(group)
    v2_unk = (group['lob_v2'] == 'unknown').sum()
    v3_unk = (group['lob_v3'] == 'unknown').sum()
    v2_pct = 100 * v2_unk / total
    v3_pct = 100 * v3_unk / total
    delta = v3_unk - v2_unk
    target_met = '✓' if v3_pct < 10 else '✗'
    print(f"{name:<22} {total:>6} {v2_unk:>8} {v2_pct:>6.1f}% {v3_unk:>8} {v3_pct:>6.1f}% {delta:>+7} {target_met:>6}")

print("-" * 82)
total = len(all_v3)
v2_unk = (all_v3['lob_v2'] == 'unknown').sum()
v3_unk = (all_v3['lob_v3'] == 'unknown').sum()
v2_pct = 100 * v2_unk / total
v3_pct = 100 * v3_unk / total
print(f"{'TOTAL':<22} {total:>6} {v2_unk:>8} {v2_pct:>6.1f}% {v3_unk:>8} {v3_pct:>6.1f}% {v3_unk-v2_unk:>+7}")

# --- 2. v3 rule distribution across full dataset ---

print("\n=== v3 rule distribution (full dataset) ===\n")
print(all_v3['lob_rule_v3'].value_counts().to_string())

# --- 3. v3 LOB distribution ---

print("\n=== v3 LOB distribution (full dataset) ===\n")
print(all_v3['lob_v3'].value_counts().to_string())

# --- 4. v3 unknown residual — print + write to file ---

import os
os.makedirs('outputs', exist_ok=True)

unknown_rows = all_v3[all_v3['lob_v3'] == 'unknown']
print(f"\n=== v3 unknown residual: {len(unknown_rows)} rows ===\n")

with open('outputs/unknown_payers_v3.txt', 'w', encoding='utf-8') as f:
    for name, group in all_v3.groupby('hospital'):
        hosp_unk = group[group['lob_v3'] == 'unknown']
        if len(hosp_unk) == 0:
            continue
        header = f"=== {name} ({len(hosp_unk)} unknowns) ==="
        print(header)
        f.write(header + "\n")
        grouped = (hosp_unk.groupby(['payer_group', 'payer_name'])
                           .size()
                           .reset_index(name='n')
                           .sort_values('n', ascending=False))
        for _, row in grouped.iterrows():
            line = f"  {row['n']:>3}  [{str(row['payer_group']):>10}]  {row['payer_name']}"
            print(line)
            f.write(line + "\n")
        print()
        f.write("\n")

print(f"Written to outputs/unknown_payers_v3.txt")

# --- 5. Quick sanity check: rows where v3 disagreed with v2 ---

disagreements = all_v3[all_v3['lob_v2'] != all_v3['lob_v3']]
print(f"\n=== v2 → v3 reclassifications: {len(disagreements)} rows ===")
print("\n(Sample of unique v2→v3 transitions, with rule that fired)")
transitions = (disagreements.groupby(['lob_v2', 'lob_v3', 'lob_rule_v3'])
                            .size()
                            .reset_index(name='n')
                            .sort_values('n', ascending=False))
print(transitions.to_string(index=False))

=== Per-hospital unknown counts: v2 vs v3 ===

hospital                total   v2_unk    v2_%   v3_unk    v3_%   delta  <10%?
----------------------------------------------------------------------------------
baylor                     45       13   28.9%        1    2.2%     -12      ✓
medical_city_alliance      46       12   26.1%        2    4.3%     -10      ✓
methodist                  82       27   32.9%        4    4.9%     -23      ✓
parkland                   65        6    9.2%        0    0.0%      -6      ✓
texas_health_plano         40        1    2.5%        0    0.0%      -1      ✓
----------------------------------------------------------------------------------
TOTAL                     278       59   21.2%        7    2.5%     -52

=== v3 rule distribution (full dataset) ===

lob_rule_v3
plan_medicare                            32
payer_group_commercial_bcbs              23
plan_chip                                23
payer_group_commercial_aetna             23
plan_ne

In [23]:
unknown_rows = all_v3[all_v3['lob_v3'] == 'unknown']
for _, row in unknown_rows.iterrows():
    print(f"  {row['hospital']:<25} payer={repr(row['payer_name']):<45} plan={repr(row['plan_name']):<50} pg={repr(row['payer_group'])}")

  baylor                    payer='QuickTrip'                                   plan='Employee Benefit Plan'                            pg='Other'
  methodist                 payer='SANA BENEFITS [2069]'                        plan='MHS HB SANA MDMC'                                 pg='Other'
  methodist                 payer='ALLIED NATIONAL [2031]'                      plan='MHS HB 90 DEGREE MDMC'                            pg='Other'
  methodist                 payer='VIBRA SPECIALITY [9216]'                     plan='MHS HB VIBRA MRMC MCMC MDMC'                      pg='Other'
  methodist                 payer='HEALTH PLANS INC [5017]'                     plan='MHS HB EMPLOYERS HEALTH NETWORK MDMC'             pg='Other'
  medical_city_alliance     payer='"Molina"'                                    plan='"STARKIDS"'                                       pg='Molina'
  medical_city_alliance     payer='"Molina"'                                    plan='"MCDSTAR"'                     

In [24]:
import os
print(os.path.exists('outputs/unknown_payers_v3.txt'))
with open('outputs/unknown_payers_v3.txt') as f:
    print(f.read())

True
=== baylor (1 unknowns) ===
    1  [     Other]  QuickTrip

=== medical_city_alliance (2 unknowns) ===
    2  [    Molina]  "Molina"

=== methodist (4 unknowns) ===
    1  [     Other]  ALLIED NATIONAL [2031]
    1  [     Other]  HEALTH PLANS INC [5017]
    1  [     Other]  SANA BENEFITS [2069]
    1  [     Other]  VIBRA SPECIALITY [9216]




In [25]:
import pandas as pd
from src.queries import classify_plan_v3

for desc, payer_name, plan_name, payer_group, expected_lob, expected_rule_substr in v3_test_cases:
    row = pd.Series({
        'payer_name': payer_name,
        'plan_name': plan_name,
        'payer_group': payer_group,
    })
    actual_lob, actual_rule = classify_plan_v3(row)
    lob_match = (actual_lob == expected_lob)
    rule_match = (expected_rule_substr in actual_rule)
    passed = lob_match and rule_match
    is_known_fn = "known FN" in desc
    
    if not passed and not is_known_fn:
        print(f"[UNEXPECTED FAIL]")
        print(f"  desc: {desc}")
        print(f"  payer_name:  {payer_name!r}")
        print(f"  plan_name:   {plan_name!r}")
        print(f"  payer_group: {payer_group!r}")
        print(f"  expected: ({expected_lob}, *{expected_rule_substr}*)")
        print(f"  actual:   ({actual_lob}, {actual_rule})")
        print()

[UNEXPECTED FAIL]
  desc: v3 new: Methodist WELLPOINT MARKETPLACE → aca_exchange
  payer_name:  'WELLPOINT MARKETPLACE EXCHANGE [3071]'
  plan_name:   'MHS HB WELLPOINT MARKETPLACE MDMC'
  payer_group: 'Other'
  expected: (aca_exchange, *payer_name_aca_exchange*)
  actual:   (aca_exchange, plan_aca)

